In [1]:
import pyNUISANCE as pn

In [2]:
input_vectors = {
    -14: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_numubar.hepmc3.gz"),
    -12: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_nuebar.hepmc3.gz"),
    12: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_nue.hepmc3.gz"),
    14: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_numu.hepmc3.gz")
}

for nupdg, evs in input_vectors.items():
    if not evs:
        raise RuntimeError(f"Failed to read event file for {nupdg}")

In [3]:
from pyProSelecta import event, part, unit, pdg, p3mod, momentum, kinetic_energy, ext
from pyNUISANCE import nhm
import numpy as np

def GetPrimaryLepton(ev):
    beam_part = nhm.EventUtils.GetBeamParticle(ev)
    prim_vertex = nhm.EventUtils.GetPrimaryVertex(ev)
    for part in prim_vertex.particles_out():
        if (part.pid() == beam_part.pid()) or\
           ( (abs(part.pid()) + 1 ) == abs(beam_part.pid())):
          return part
    raise runtime_error("Failed to find primary lepton out: %s -> [%s]" % \
                        (beam_part.pid(), ",".join(part.pid for part in prim_vertex.particles_out())))

def PrimaryLeptonTotalEnergy(ev):
    pl = GetPrimaryLepton(ev)
    return (pl.momentum().e() / unit.GeV_c) if pl.pid() % 2 else 0

def NeutralPionTotalEnergy(ev):
    return part.sum(momentum, event.all_out_part(ev, 111)).e() / unit.GeV_c

def NumberOfChargedPions(ev):
    return event.num_out_part(ev, [211,-211], flatten=True)

def ChargedPionKineticEnergy(ev):
    return part.sum(kinetic_energy, event.all_out_part(ev, [211, -211], flatten=True)) / unit.GeV_c

def ProtonKineticEnergy(ev):
    return part.sum(kinetic_energy, event.all_out_part(ev, 2212)) / unit.GeV_c

def ReconstructedNeutrinoEnergy_true(ev):
    return PrimaryLeptonTotalEnergy(ev) + ChargedPionKineticEnergy(ev) + (NumberOfChargedPions(ev)*0.1395) +  NeutralPionTotalEnergy(ev) + ProtonKineticEnergy(ev)

event_frames = {}

for nupdg, evs in input_vectors.items():
    event_frames[nupdg] = pn.EventFrameGen(evs) \
                            .add_column("pid_neutrino", lambda ev: nhm.EventUtils.GetBeamParticle(ev).pid()) \
                            .add_column("E_neutrino", ext.enu_GeV) \
                            .add_column("is_CC", ext.isCC) \
                            .add_column("pid_lepton", lambda ev: GetPrimaryLepton(ev).pid()) \
                            .add_column("E_lepton", PrimaryLeptonTotalEnergy) \
                            .add_column("T_proton", ProtonKineticEnergy) \
                            .add_column("num_pi0", lambda ev: event.num_out_part(ev, 111)) \
                            .add_column("E_pi0", NeutralPionTotalEnergy) \
                            .add_column("num_cpi", NumberOfChargedPions) \
                            .add_column("T_cpi", ChargedPionKineticEnergy) \
                            .add_column("E_nu_rec_true", ReconstructedNeutrinoEnergy_true).all()

output_columns = [
    "pid_neutrino",
    "E_neutrino",
    "is_CC",
    "pid_lepton",
    "E_lepton",
    "T_proton",
    "num_pi0",
    "E_pi0",
    "num_cpi",
    "T_cpi",
    "E_nu_rec_true" ]

print(event_frames[14])

 ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 | event.number | weight.cv | fatx_per_su$ | fatx_per_su$ | process.id | pid_neutrino | E_neutrino | is_CC | pid_lepton | E_lepton | T_proton | num_pi0 | E_pi0 | num_cpi |   T_cpi | E_nu_rec_tr$ |
 ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 |            0 |         1 |    1.374e-06 |    3.434e-08 |        500 |           14 |      3.222 |     1 |         13 |    2.019 |   0.3915 |       0 |     0 |       2 |  0.5405 |         3.23 |
 |            1 |         1 |    6.869e-07 |    1.717e-08 |        251 |           14 |        3.2 |     0 |         14 |        0 |        0 |       0 |     0 |       0 |       0 |            0 |
 |            2

In [6]:
import pandas as pa

def todf(ef, index):
  df = pa.DataFrame(data=ef.table,
                    index=index,
                    columns=ef.column_names)
  for row in ["event.number", "process.id", "pid_neutrino", "is_CC", "pid_lepton", "num_pi0", "num_cpi"]:
    df[row] = pa.to_numeric(df[row], downcast="integer")
  return df

In [7]:
df_numu = todf(event_frames[14], [i for i in range(event_frames[14].table.shape[0])])
df_nue = todf(event_frames[12], [i + event_frames[14].table.shape[0] for i in range(event_frames[12].table.shape[0])])
df_numode = pa.concat([df_numu, df_nue])
df_numode = df_numode.sample(frac=1)
df_numode

,event.number,weight.cv,fatx_per_sumw.pb_per_target.estimate,fatx_per_sumw.pb_per_nucleon.estimate,process.id,pid_neutrino,E_neutrino,is_CC,pid_lepton,E_lepton,T_proton,num_pi0,E_pi0,num_cpi,T_cpi,E_nu_rec_true
79046,79046,1.0,1.737896e-11,4.344740e-13,300,14,2.548099,1,13,2.089750,0.079629,0,0.000000,0,0.000000,2.169379
119433,19433,1.0,7.194585e-11,1.798646e-12,600,12,9.635826,1,11,2.870737,0.133741,1,0.267034,0,0.000000,3.271511
15484,15484,1.0,8.871518e-11,2.217879e-12,402,14,2.044454,1,13,1.739221,0.093686,0,0.000000,0,0.000000,1.832907
84504,84504,1.0,1.625649e-11,4.064122e-13,451,14,4.203669,0,14,0.000000,0.288874,0,0.000000,0,0.000000,0.288874
141753,41753,1.0,3.348651e-11,8.371628e-13,400,12,1.257986,1,11,0.226893,0.297321,0,0.000000,1,0.035920,0.699634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191131,91131,1.0,1.534253e-11,3.835633e-13,500,12,3.900935,1,11,2.164597,0.790263,1,0.728289,0,0.000000,3.683148
134609,34609,1.0,4.039861e-11,1.009965e-12,200,12,2.968862,1,11,2.194017,0.487557,0,0.000000,0,0.000000,2.681574
192913,92913,1.0,1.504828e-11,3.762070e-13,600,12,3.782191,1,11,0.286570,0.118337,1,3.384344,0,0.000000,3.789251
62886,62886,1.0,2.184481e-11,5.461202e-13,400,14,2.791902,1,13,1.045507,1.138378,1,0.204531,2,0.095965,2.763382


In [8]:
df_numode.to_csv("neutrino_mode_events.csv",index_label="event_num",columns=output_columns, float_format="%.4f")

In [9]:
df_numubar = todf(event_frames[-14], [i for i in range(event_frames[-14].table.shape[0])])
df_nuebar = todf(event_frames[-12], [i + event_frames[-14].table.shape[0] for i in range(event_frames[12].table.shape[0])])
df_nubarmode = pa.concat([df_numubar, df_nuebar])
df_nubarmode = df_nubarmode.sample(frac=1).reset_index(drop=True)
df_nubarmode

,event.number,weight.cv,fatx_per_sumw.pb_per_target.estimate,fatx_per_sumw.pb_per_nucleon.estimate,process.id,pid_neutrino,E_neutrino,is_CC,pid_lepton,E_lepton,T_proton,num_pi0,E_pi0,num_cpi,T_cpi,E_nu_rec_true
0,63996,1.0,8.499085e-12,2.124771e-13,225,-12,2.964859,1,-11,2.792870,0.000000,0,0.000000,0,0.000000,2.792870
1,31050,1.0,1.722221e-11,4.305552e-13,225,-14,3.226975,1,-13,2.581443,0.137801,0,0.000000,0,0.000000,2.719243
2,88452,1.0,6.045774e-12,1.511443e-13,425,-14,1.726876,1,-13,1.063710,0.187940,0,0.000000,1,0.091966,1.483115
3,65712,1.0,8.137915e-12,2.034479e-13,225,-14,1.634292,1,-13,1.492622,0.039105,0,0.000000,0,0.000000,1.531728
4,19912,1.0,2.731461e-11,6.828654e-13,426,-12,2.442468,1,-11,1.196985,0.000000,1,1.186701,0,0.000000,2.383686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,60082,1.0,9.052742e-12,2.263186e-13,525,-12,1.395124,1,-11,0.644535,0.384652,1,0.159929,1,0.021531,1.350148
199996,15714,1.0,3.402907e-11,8.507267e-13,675,-14,33.696258,0,-14,0.000000,5.314237,0,0.000000,6,13.753359,19.904596
199997,97118,1.0,5.600510e-12,1.400127e-13,625,-12,4.128943,1,-11,1.216583,0.000000,1,0.374324,3,0.613128,2.622535
199998,55691,1.0,9.602220e-12,2.400555e-13,426,-14,2.657956,1,-13,1.474363,0.037263,0,0.000000,2,0.145300,1.935926


In [10]:
df_nubarmode.to_csv("antineutrino_mode_events.csv",index_label="event_num",columns=output_columns, float_format="%.4f")